# 01 — Data Exploration

First contact with the raw **IBM Telco Customer Churn** dataset:
schema, data types, summary statistics, missing values and value ranges. All loading logic is reused from `src/data_preprocessing.py` so the notebook stays in sync with the production code.

In [1]:
import sys
sys.path.insert(0, 'C:\\Users\\OUISSAL\\Production-Ready Customer Churn')  # make 'src' importable
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import pandas as pd
from src.data_preprocessing import load_raw_data

df = load_raw_data()
print(f'Shape: {df.shape}')

Shape: (7043, 21)


## 1. First look at the data

In [2]:
df.head(8)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No


## 2. Column names and data types

In [3]:
pd.DataFrame({'dtype': df.dtypes, 'non_null': df.notna().sum()})

,dtype,non_null
customerID,str,7043
gender,str,7043
SeniorCitizen,int64,7043
Partner,str,7043
Dependents,str,7043
tenure,int64,7043
PhoneService,str,7043
MultipleLines,str,7043
InternetService,str,7043
OnlineSecurity,str,7043


## 3. Summary statistics (numerical columns)

In [4]:
df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]

,count,mean,std,min,50%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,29.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,70.35,118.75


## 4. Missing values

`TotalCharges` is stored as text and contains 11 blank values — these correspond to brand-new customers (`tenure == 0`).

In [5]:
missing = (df.isna().sum()
           + df.astype(str).apply(lambda s: s.str.strip() == '').sum())
print(missing[missing > 0])

TotalCharges    11
dtype: int64


## 5. Value cardinality per categorical column

In [6]:
for col in df.select_dtypes(include='object').columns:
    values = df[col].dropna().unique()
    if len(values) <= 6:
        print(f'{col:<22} {len(values):>2} values: {list(values)}')
    else:
        print(f'{col:<22} {len(values):>2} values '
              f'(high cardinality, likely an ID)')

customerID             7043 values (high cardinality, likely an ID)
gender                  2 values: ['Female', 'Male']
Partner                 2 values: ['Yes', 'No']
Dependents              2 values: ['No', 'Yes']
PhoneService            2 values: ['No', 'Yes']
MultipleLines           3 values: ['No phone service', 'No', 'Yes']
InternetService         3 values: ['DSL', 'Fiber optic', 'No']
OnlineSecurity          3 values: ['No', 'Yes', 'No internet service']
OnlineBackup            3 values: ['Yes', 'No', 'No internet service']
DeviceProtection        3 values: ['No', 'Yes', 'No internet service']
TechSupport             3 values: ['No', 'Yes', 'No internet service']
StreamingTV             3 values: ['No', 'Yes', 'No internet service']
StreamingMovies         3 values: ['No', 'Yes', 'No internet service']
Contract                3 values: ['Month-to-month', 'One year', 'Two year']
PaperlessBilling        2 values: ['Yes', 'No']
PaymentMethod           4 values: ['Electronic check'

**Key observations**

* 7,043 rows, 21 columns — small, clean and realistic.
* `customerID` is an identifier, not a feature.
* `TotalCharges` is text and needs coercion; 11 blanks for new customers.
* `SeniorCitizen` is stored as 0/1 while all other flags use Yes/No.
* No other missing values — the dataset is mostly clean.